# Music Annotation Project V7.3: Leakage-Safe Paper/Telea Router Benchmark

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook reuses completed V7.1 Phase-A reconstruction caches at dilation 2. It does not rerun the expensive V7.1 reconstruction suite.

Primary comparison:
- cached plain `telea` baseline;
- `paper_telea_router_r5`;
- `paper_telea_router_r9`.

Leakage controls:
- router construction receives only the observed page and cached removal mask;
- masked pixels are whitened before visible-ink and staff evidence are detected;
- the clean page is loaded only after all candidate reconstructions and route masks have been created;
- pixels outside the removal mask are asserted unchanged.

Predefined exploratory production gates:
- background dark-smudge benefit versus Telea has a positive 95% bootstrap lower bound;
- staff ink-recall delta is at least -0.005;
- symbol edge-F1 delta is at least -0.010;
- mixed edge-F1 delta is at least -0.010;
- background-smudge wins exceed losses;
- paper-route structure leakage is at most 5%.

Passing these gates supports a provisional candidate, not independent confirmation. The same controlled pages are used to compare the two fixed radii.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas matplotlib


## Configuration and V7.1 cache validation


In [ ]:
import cv2, json, hashlib, shutil, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from skimage.metrics import structural_similarity as ssim

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'
OBSERVED_DIR=ROOT/'aligned'
V71_ROOT=ROOT/'reconstruction_benchmark_v7_1_real'
STATE_PATH=V71_ROOT/'benchmark_state.json'
OUTPUT_ROOT=ROOT/'reconstruction_router_benchmark_v7_3'
PERSISTENT_CACHE=OUTPUT_ROOT/'persistent_cache'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=OUTPUT_ROOT/'runs'/(RUN_NAME+'_v7_3')
TABLES=OUT/'tables'; ROUTE_MAPS=OUT/'route_maps'; CONTACTS=OUT/'contact_sheets'; FAILURES=OUT/'failure_crops'
LOCAL=Path('/content/reconstruction_router_v7_3'); LCLEAN=LOCAL/'clean'; LOBS=LOCAL/'observed'
for d in [OUTPUT_ROOT,PERSISTENT_CACHE,OUT,TABLES,ROUTE_MAPS,CONTACTS,FAILURES,LCLEAN,LOBS]: d.mkdir(parents=True,exist_ok=True)

DILATION=2
RADII=[5,9]
METHODS=['telea','paper_telea_router_r5','paper_telea_router_r9']
ROUTERS=METHODS[1:]
CROP_MARGIN=48
MIN_COMPONENT_AREA=20
BOOTSTRAP_SAMPLES=2000
MAX_CONTACT_PAGES=24
MAX_FAILURE_CROPS=20
FORCE_REBUILD=False
ROUTER_SCHEMA='v7.3_visible_ink_plus_staff_guard_hard_route'
GATES={'staff_ink_recall_min_delta':-0.005,'symbol_edge_f1_min_delta':-0.010,
       'mixed_edge_f1_min_delta':-0.010,'paper_route_structure_leakage_max':0.05}

def image_files(folder):
    folder=Path(folder)
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if folder.exists() else []
def maps(folder): return {p.stem:p for p in image_files(folder)}
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def object_hash(value): return hashlib.md5(json.dumps(value,sort_keys=True).encode()).hexdigest()
def read_json(path,default=None):
    try: return json.loads(Path(path).read_text()) if Path(path).exists() else default
    except Exception: return default
def write_json_atomic(path,value):
    path=Path(path); temp=Path(str(path)+'.tmp')
    temp.write_text(json.dumps(value,indent=2,allow_nan=True,default=str)); temp.replace(path)
def valid_npz(path,required):
    try:
        with np.load(path) as z: return set(required).issubset(z.files)
    except Exception: return False
def copy_current(source,destination):
    destination=Path(destination)
    if not destination.exists() or md5(source)!=md5(destination): shutil.copy2(source,destination)
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0])
    return out

state=read_json(STATE_PATH)
if not state or not isinstance(state.get('items'),dict): raise RuntimeError(f'Invalid V7.1 state: {STATE_PATH}')
clean_map=maps(CLEAN_DIR); observed_map=maps(OBSERVED_DIR)
valid_pages=[]; issues=[]
for key,item in sorted(state['items'].items()):
    if int(item.get('dilation',-1))!=DILATION: continue
    page=item.get('page')
    if not page or page not in clean_map or page not in observed_map:
        issues.append({'item_id':key,'page':page,'reason':'clean or observed source missing'}); continue
    cache=Path(item.get('reconstruction_cache',''))
    if not item.get('reconstruction_complete') or not valid_npz(cache,['mask','telea']):
        issues.append({'item_id':key,'page':page,'reason':'valid V7.1 d2 cache with mask and telea missing'}); continue
    clean=cv2.imread(str(clean_map[page]),0); observed=cv2.imread(str(observed_map[page]),0)
    if clean is None or observed is None or clean.shape!=observed.shape:
        issues.append({'item_id':key,'page':page,'reason':'source unreadable or shape mismatch'}); continue
    with np.load(cache) as z: cache_shape=z['mask'].shape
    if cache_shape!=observed.shape:
        issues.append({'item_id':key,'page':page,'reason':'cache mask/source shape mismatch'}); continue
    copy_current(clean_map[page],LCLEAN/(page+clean_map[page].suffix.lower()))
    copy_current(observed_map[page],LOBS/(page+observed_map[page].suffix.lower()))
    valid_pages.append(page)
valid_pages=sorted(set(valid_pages))
pd.DataFrame(issues).to_csv(TABLES/'discovery_issues.csv',index=False)
if not valid_pages: raise RuntimeError('No reusable V7.1 dilation-2 caches were found.')
print('Reusable V7.1 d2 pages:',len(valid_pages),'Issues:',len(issues))


## Leakage-safe routing and reconstruction functions


In [ ]:
def find_local(folder,page):
    hits=[p for p in image_files(folder) if p.stem.lower()==page.lower()]
    return hits[0] if len(hits)==1 else None

def padded_components(mask,margin=CROP_MARGIN,min_area=8):
    n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    boxes=[]; h,w=mask.shape
    for i in range(1,n):
        x,y,bw,bh,area=stats[i]
        if area>=min_area: boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0); ay0=min(ay0,by0); ax1=max(ax1,bx1); ay1=max(ay1,by1); changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def paper_fill(observed,mask):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        image=observed[y0:y1,x0:x1]; local_mask=mask[y0:y1,x0:x1]>0
        ring=cv2.dilate(local_mask.astype(np.uint8),np.ones((21,21),np.uint8))>0
        ring &= ~local_mask
        bright=image[ring & (image>=180)]
        if bright.size<25: bright=image[~local_mask]
        value=int(np.median(bright)) if bright.size else 255
        local=result[y0:y1,x0:x1]; local[local_mask]=value; result[y0:y1,x0:x1]=local
    return immutable(observed,result,mask)

def visible_ink_evidence(observed,mask):
    visible=observed.copy(); visible[mask>0]=255
    return cv2.adaptiveThreshold(visible,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def staff_evidence(observed,mask):
    visible=observed.copy(); visible[mask>0]=255
    ink=(visible<190).astype(np.uint8)*255
    long_lines=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))
    return cv2.dilate(long_lines,np.ones((3,7),np.uint8))>0

def route_mask(observed,mask,radius):
    visible_ink=visible_ink_evidence(observed,mask)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*radius+1,2*radius+1))
    near_ink=cv2.dilate(visible_ink.astype(np.uint8),k)>0
    staff_guard=cv2.dilate(staff_evidence(observed,mask).astype(np.uint8),np.ones((7,11),np.uint8))>0
    structure=(mask>0)&(near_ink|staff_guard)
    paper=(mask>0)&~structure
    return paper,structure

def routed_candidate(observed,mask,telea,paper,radius):
    paper_route,structure_route=route_mask(observed,mask,radius)
    routed=observed.copy(); routed[paper_route]=paper[paper_route]; routed[structure_route]=telea[structure_route]
    return immutable(observed,routed,mask),paper_route,structure_route

def route_visual(observed,mask,paper_route,structure_route):
    base=cv2.cvtColor(observed,cv2.COLOR_GRAY2BGR); layer=base.copy()
    layer[paper_route]=(0,190,0); layer[structure_route]=(255,90,0)
    boundary=cv2.morphologyEx((mask>0).astype(np.uint8),cv2.MORPH_GRADIENT,np.ones((3,3),np.uint8))>0
    layer[boundary]=(0,0,255)
    return cv2.addWeighted(base,.62,layer,.38,0)


## Construct candidates before clean-page scoring


In [ ]:
reconstruction_records=[]; routing_records=[]
for index,page in enumerate(valid_pages,1):
    observed=cv2.imread(str(find_local(LOBS,page)),0)
    item=state['items'][f'{page}|d{DILATION}']
    v71_cache=Path(item['reconstruction_cache'])
    with np.load(v71_cache) as z: mask=z['mask']; telea=z['telea']
    if not np.array_equal(telea[mask==0],observed[mask==0]): raise RuntimeError(f'{page}: cached Telea changed unmasked pixels')
    fingerprint=object_hash({'v71_reconstruction_fingerprint':item.get('fingerprint'),'observed_md5':md5(find_local(LOBS,page)),
        'schema':ROUTER_SCHEMA,'radii':RADII,'margin':CROP_MARGIN})
    cache=PERSISTENT_CACHE/f'{page}_d{DILATION}_{fingerprint}.npz'
    if cache.exists() and not FORCE_REBUILD and valid_npz(cache,['mask','telea','paper_telea_router_r5','paper_telea_router_r9','paper_route_r5','paper_route_r9']):
        with np.load(cache) as z:
            outputs={m:z[m] for m in METHODS}; routes={r:z[f'paper_route_r{r}']>0 for r in RADII}
        hit=True
    else:
        paper=paper_fill(observed,mask)
        outputs={'telea':telea}; routes={}; structures={}
        for radius in RADII:
            candidate,paper_route,structure_route=routed_candidate(observed,mask,telea,paper,radius)
            outputs[f'paper_telea_router_r{radius}']=candidate; routes[radius]=paper_route; structures[radius]=structure_route
        np.savez_compressed(cache,mask=mask,telea=telea,
            paper_telea_router_r5=outputs['paper_telea_router_r5'],paper_telea_router_r9=outputs['paper_telea_router_r9'],
            paper_route_r5=routes[5].astype(np.uint8),paper_route_r9=routes[9].astype(np.uint8))
        hit=False
    for method,image in outputs.items():
        if not np.array_equal(image[mask==0],observed[mask==0]): raise RuntimeError(f'{page} {method}: outside-mask drift')
    for radius in RADII:
        paper_route=routes[radius]; structure_route=(mask>0)&~paper_route
        routing_records.append({'page':page,'radius':radius,'mask_pixels':int(np.sum(mask>0)),
            'paper_routed_pixels':int(paper_route.sum()),'telea_routed_pixels':int(structure_route.sum()),
            'paper_route_percent':float(100*paper_route.sum()/max(np.sum(mask>0),1)),'router_cache':str(cache)})
        cv2.imwrite(str(ROUTE_MAPS/f'{page}_r{radius}.png'),route_visual(observed,mask,paper_route,structure_route))
    reconstruction_records.append({'page':page,'cache':str(cache),'cache_hit':hit})
    print(f'Router construction {index}/{len(valid_pages)} {page}:', 'HIT' if hit else 'MISS')

pd.DataFrame(routing_records).to_csv(TABLES/'routing_by_page_pre_scoring.csv',index=False)
pd.DataFrame(reconstruction_records).to_csv(TABLES/'reconstruction_cache_status.csv',index=False)
print('All candidates constructed without loading clean pages in this cell.')


## Evaluation-only scoring, routing confusion and components


In [ ]:
def printed_ink(image): return cv2.adaptiveThreshold(image,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edge_map(image): return cv2.Canny(image,70,170)>0
def classify_gt(clean,mask):
    ink=printed_ink(clean); long_lines=cv2.morphologyEx(ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long_lines.astype(np.uint8),np.ones((3,7),np.uint8))>0
    near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0; m=mask>0
    return {'all':m,'background':m&~ink,'staff':m&staff,'symbol':m&ink&~near,'mixed':m&ink&near},ink

def tolerant_f1(reference,prediction,region,tolerance=1):
    reference=reference&region; prediction=prediction&region
    if not reference.sum(): return np.nan,np.nan,np.nan
    k=np.ones((2*tolerance+1,2*tolerance+1),np.uint8)
    rd=cv2.dilate(reference.astype(np.uint8),k)>0; pdil=cv2.dilate(prediction.astype(np.uint8),k)>0
    precision=float(np.sum(prediction&rd)/max(prediction.sum(),1)); recall=float(np.sum(reference&pdil)/reference.sum())
    return precision,recall,float(2*precision*recall/max(precision+recall,1e-12))
def metrics(clean,recon,region):
    if not region.sum(): return None
    diff=clean.astype(np.float32)-recon.astype(np.float32); _,sm=ssim(clean,recon,data_range=255,full=True)
    mse=float(np.mean((diff[region])**2)); ip,ir,if1=tolerant_f1(printed_ink(clean),printed_ink(recon),region); _,_,ef1=tolerant_f1(edge_map(clean),edge_map(recon),region)
    smudge=(recon<180)&(clean>=220)
    return {'pixels':int(region.sum()),'mae':float(np.mean(np.abs(diff[region]))),'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),
        'ssim':float(np.mean(sm[region])),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1_tolerant':ef1,
        'dark_smudge_rate':float(np.sum(smudge&region)/max(np.sum((clean>=220)&region),1))}

rows=[]; components=[]; route_pages=[]; route_components=[]
for index,page in enumerate(valid_pages,1):
    clean=cv2.imread(str(find_local(LCLEAN,page)),0); observed=cv2.imread(str(find_local(LOBS,page)),0)
    record=next(x for x in reconstruction_records if x['page']==page)
    with np.load(record['cache']) as z:
        mask=z['mask']; outputs={m:z[m] for m in METHODS}; routes={r:z[f'paper_route_r{r}']>0 for r in RADII}
    regions,clean_ink=classify_gt(clean,mask)
    for method,recon in outputs.items():
        for region_name,region in regions.items():
            q=metrics(clean,recon,region)
            if q: rows.append({'page':page,'dilation':DILATION,'method':method,'region':region_name,**q})
    n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    for radius in RADII:
        paper_route=routes[radius]; background=regions['background']; structure=(mask>0)&clean_ink
        paper_background=int(np.sum(paper_route&background)); paper_structure=int(np.sum(paper_route&structure)); paper_total=int(paper_route.sum())
        route_pages.append({'page':page,'radius':radius,'mask_pixels':int(np.sum(mask>0)),'paper_routed_pixels':paper_total,
            'telea_routed_pixels':int(np.sum((mask>0)&~paper_route)),'paper_route_percent':float(100*paper_total/max(np.sum(mask>0),1)),
            'clean_background_pixels_sent_to_paper':paper_background,'clean_structure_pixels_sent_to_paper':paper_structure,
            'paper_route_background_precision':float(paper_background/max(paper_total,1)),
            'paper_route_structure_leakage':float(paper_structure/max(paper_total,1))})
    for component in range(1,n):
        area=int(stats[component,cv2.CC_STAT_AREA])
        if area<MIN_COMPONENT_AREA: continue
        x,y,w,h=map(int,stats[component,:4]); comp=labels==component; ink_fraction=float(np.mean(clean_ink[comp]))
        category='background' if ink_fraction<.05 else ('structure' if ink_fraction>.30 else 'mixed')
        for method,recon in outputs.items():
            q=metrics(clean,recon,comp)
            if q: components.append({'page':page,'component':component,'category':category,'component_area':area,'x':x,'y':y,'w':w,'h':h,
                'clean_ink_fraction':ink_fraction,'method':method,**q})
        for radius in RADII:
            paper_component=routes[radius]&comp; ptotal=int(paper_component.sum()); pstructure=int(np.sum(paper_component&clean_ink))
            route_components.append({'page':page,'component':component,'category':category,'radius':radius,'component_area':area,
                'paper_routed_pixels':ptotal,'paper_route_percent':float(100*ptotal/max(area,1)),
                'paper_route_structure_leakage':float(pstructure/max(ptotal,1))})
    print(f'Evaluation {index}/{len(valid_pages)} {page}')

results=pd.DataFrame(rows); component_df=pd.DataFrame(components); routing_df=pd.DataFrame(route_pages); routing_component_df=pd.DataFrame(route_components)
results.to_csv(TABLES/'metrics_by_page_region.csv',index=False); component_df.to_csv(TABLES/'component_metrics.csv',index=False)
routing_df.to_csv(TABLES/'routing_by_page.csv',index=False); routing_component_df.to_csv(TABLES/'routing_by_component.csv',index=False)
print('Metric rows:',len(results),'Component rows:',len(component_df))


## Paired Telea comparisons and production gates


In [ ]:
def bootstrap_delta(values,seed=42):
    v=np.asarray(values,float); v=v[np.isfinite(v)]
    if not len(v): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed); samples=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(BOOTSTRAP_SAMPLES)])
    return float(v.mean()),float(np.percentile(samples,2.5)),float(np.percentile(samples,97.5))

comparisons=[]
comparison_specs=[('background','dark_smudge_rate',False),('background','mae',False),('staff','ink_recall',True),
                  ('symbol','edge_f1_tolerant',True),('mixed','edge_f1_tolerant',True),('all','mae',False),('all','ssim',True)]
for region,metric,higher in comparison_specs:
    pivot=results[results.region==region].pivot(index='page',columns='method',values=metric)
    if 'telea' not in pivot: continue
    for method in ROUTERS:
        delta=(pivot[method]-pivot['telea'])*(1 if higher else -1)
        mean,low,high=bootstrap_delta(delta.dropna())
        comparisons.append({'region':region,'metric':metric,'method':method,'baseline':'telea','benefit_mean':mean,
            'ci95_low':low,'ci95_high':high,'wins':int((delta>0).sum()),'losses':int((delta<0).sum()),'ties':int((delta==0).sum()),'pages':int(delta.notna().sum())})
paired=pd.DataFrame(comparisons); paired.to_csv(TABLES/'paired_vs_telea.csv',index=False)

route_summary=(routing_df.groupby('radius',as_index=False).agg(pages=('page','nunique'),mask_pixels=('mask_pixels','sum'),paper_routed_pixels=('paper_routed_pixels','sum'),
    background_sent_to_paper=('clean_background_pixels_sent_to_paper','sum'),structure_sent_to_paper=('clean_structure_pixels_sent_to_paper','sum')))
route_summary['paper_route_percent']=100*route_summary.paper_routed_pixels/route_summary.mask_pixels
route_summary['paper_route_background_precision']=route_summary.background_sent_to_paper/route_summary.paper_routed_pixels.clip(lower=1)
route_summary['paper_route_structure_leakage']=route_summary.structure_sent_to_paper/route_summary.paper_routed_pixels.clip(lower=1)
route_summary.to_csv(TABLES/'routing_confusion_summary.csv',index=False)

def paired_value(method,region,metric,column):
    q=paired[(paired.method==method)&(paired.region==region)&(paired.metric==metric)]
    return float(q.iloc[0][column]) if len(q) else np.nan

gate_rows=[]
for radius in RADII:
    method=f'paper_telea_router_r{radius}'; rs=route_summary[route_summary.radius==radius].iloc[0]
    smudge_mean=paired_value(method,'background','dark_smudge_rate','benefit_mean'); smudge_low=paired_value(method,'background','dark_smudge_rate','ci95_low')
    smudge_wins=paired_value(method,'background','dark_smudge_rate','wins'); smudge_losses=paired_value(method,'background','dark_smudge_rate','losses')
    staff_delta=paired_value(method,'staff','ink_recall','benefit_mean'); symbol_delta=paired_value(method,'symbol','edge_f1_tolerant','benefit_mean'); mixed_delta=paired_value(method,'mixed','edge_f1_tolerant','benefit_mean')
    tests={'background_smudge_mean_positive':smudge_mean>0,'background_smudge_ci_low_positive':smudge_low>0,
           'background_smudge_wins_exceed_losses':smudge_wins>smudge_losses,
           'staff_noninferior':staff_delta>=GATES['staff_ink_recall_min_delta'],
           'symbol_noninferior':symbol_delta>=GATES['symbol_edge_f1_min_delta'],
           'mixed_noninferior':mixed_delta>=GATES['mixed_edge_f1_min_delta'],
           'routing_structure_leakage_acceptable':float(rs.paper_route_structure_leakage)<=GATES['paper_route_structure_leakage_max']}
    gate_rows.append({'method':method,'radius':radius,'background_smudge_benefit_mean':smudge_mean,'background_smudge_ci95_low':smudge_low,
        'background_smudge_wins':smudge_wins,'background_smudge_losses':smudge_losses,'staff_ink_recall_delta':staff_delta,
        'symbol_edge_f1_delta':symbol_delta,'mixed_edge_f1_delta':mixed_delta,'paper_route_structure_leakage':float(rs.paper_route_structure_leakage),
        **tests,'passes_all_predefined_gates':all(tests.values()),'interpretation':'exploratory; independent confirmation still required'})
gates_df=pd.DataFrame(gate_rows); gates_df.to_csv(TABLES/'production_gate.csv',index=False)
display(paired); display(route_summary); display(gates_df)


## Contact sheets and category-specific failure crops


In [ ]:
summary=(results.groupby(['method','region'],as_index=False).agg(pages=('page','nunique'),pixels=('pixels','sum'),mae=('mae','mean'),psnr=('psnr','mean'),
    ssim=('ssim','mean'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1_tolerant=('edge_f1_tolerant','mean'),dark_smudge_rate=('dark_smudge_rate','mean')))
summary.to_csv(TABLES/'summary_by_region.csv',index=False)
page_priority=(routing_df.groupby('page').paper_routed_pixels.max().sort_values(ascending=False).head(MAX_CONTACT_PAGES).index)
for page in page_priority:
    clean=cv2.imread(str(find_local(LCLEAN,page)),0); observed=cv2.imread(str(find_local(LOBS,page)),0)
    rec=next(x for x in reconstruction_records if x['page']==page)
    with np.load(rec['cache']) as z:
        mask=z['mask']; telea=z['telea']; r5=z['paper_telea_router_r5']; r9=z['paper_telea_router_r9']; p5=z['paper_route_r5']>0; p9=z['paper_route_r9']>0
    panels=[clean,observed,route_visual(observed,mask,p5,(mask>0)&~p5),telea,r5,route_visual(observed,mask,p9,(mask>0)&~p9),r9]
    names=['clean','observed','route r5','telea','router r5','route r9','router r9']
    fig,axes=plt.subplots(2,4,figsize=(24,14))
    for ax,image,name in zip(axes.ravel(),panels,names):
        ax.imshow(image if image.ndim==2 else cv2.cvtColor(image,cv2.COLOR_BGR2RGB),cmap='gray' if image.ndim==2 else None,vmin=0 if image.ndim==2 else None,vmax=255 if image.ndim==2 else None); ax.set_title(name); ax.axis('off')
    for ax in axes.ravel()[len(panels):]: ax.axis('off')
    plt.tight_layout(); plt.savefig(CONTACTS/f'{page}_comparison.png',dpi=150,bbox_inches='tight'); plt.close(fig)

for category,metric,higher in [('background','dark_smudge_rate',False),('structure','ink_recall',True),('mixed','edge_f1_tolerant',True)]:
    folder=FAILURES/category; folder.mkdir(parents=True,exist_ok=True)
    target=component_df[(component_df.category==category)&(component_df.method=='paper_telea_router_r5')].dropna(subset=[metric]).sort_values(metric,ascending=higher).head(MAX_FAILURE_CROPS)
    for rank,(_,row) in enumerate(target.iterrows(),1):
        page=str(row.page); rec=next(x for x in reconstruction_records if x['page']==page)
        clean=cv2.imread(str(find_local(LCLEAN,page)),0); observed=cv2.imread(str(find_local(LOBS,page)),0)
        with np.load(rec['cache']) as z: images=[clean,observed,z['telea'],z['paper_telea_router_r5'],z['paper_telea_router_r9']]
        pad=28; H,W=clean.shape; x0=max(0,int(row.x)-pad); y0=max(0,int(row.y)-pad); x1=min(W,int(row.x+row.w)+pad); y1=min(H,int(row.y+row.h)+pad)
        names=['clean','observed','telea','router r5','router r9']; fig,axes=plt.subplots(1,5,figsize=(20,4))
        for ax,image,name in zip(axes,images,names): ax.imshow(image[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255); ax.set_title(name); ax.axis('off')
        plt.tight_layout(); plt.savefig(folder/f'{rank:03d}_{page}_c{int(row.component)}.png',dpi=170,bbox_inches='tight'); plt.close(fig)
print('Contact sheets:',CONTACTS,'Failure crops:',FAILURES)


## Manifest and completion audit


In [ ]:
resume=pd.DataFrame(reconstruction_records)
resume['evaluated']=resume.page.isin(results.page.unique()); resume.to_csv(TABLES/'resume_status.csv',index=False)
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.3','source_phase':'V7.1 Phase A dilation 2',
    'pages_discovered':len(valid_pages),'pages_evaluated':int(results.page.nunique()),'methods':METHODS,'radii':RADII,
    'router_schema':ROUTER_SCHEMA,'settings':{'dilation':DILATION,'crop_margin':CROP_MARGIN,'bootstrap_samples':BOOTSTRAP_SAMPLES,
        'minimum_component_area':MIN_COMPONENT_AREA,'production_gates':GATES},
    'guarantees':{'clean_unavailable_during_router_construction':True,'clean_loaded_only_in_scoring_cell':True,
        'masked_pixels_whitened_before_visible_structure_detection':True,'outside_mask_immutable_checked':True,
        'v7_1_telea_and_mask_reused':True,'routed_candidates_persistently_cached':True},
    'interpretation':'Exploratory comparison on the existing controlled pages; a passing router still requires independent confirmation on frozen settings.',
    'outputs':{'tables':str(TABLES),'route_maps':str(ROUTE_MAPS),'contact_sheets':str(CONTACTS),'failure_crops':str(FAILURES)}}
(OUT/'benchmark_manifest.json').write_text(json.dumps(manifest,indent=2))
print('V7.3 COMPLETE:',OUT)
print('Production gates:'); display(gates_df)
